# The pipeline state after preflight

Runs the **real** pipeline graph (`agents/pipeline_agent.py`) and stops it right after the `preflight` node, so you
can look at what the next node receives. Nothing is sent to a model.

| § | What | |
|---|---|---|
| 0 | Setup: the demo repository and where a credential may be found | |
| 1 | Run the graph up to and including `preflight` | |
| 2 | The state: `PipelineRun` after preflight | graph state, seen by tracers |
| 3 | What a tracer would receive for that state | the check that no secret is in it |
| 4 | The run context: `PipelineDeps` | not state; tracers never see it |
| 5 | When preflight fails | the graph routes straight to `END` |

Two places hold per-run data, and the difference matters. **State** (`PipelineRun`) is what every node reads and
writes, and what a tracer such as Langfuse serializes as each node's input and output. **Context** (`PipelineDeps`)
is what nodes may read but never write: where a credential may be found, and how to build the review port. The
credential itself is in neither. The `review` node looks it up and hands it straight to the model client.

## 0 · Setup

In [1]:
import json
import os
import subprocess
import sys
import tempfile
from pathlib import Path

PROJECT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT))

# ---- change these ---------------------------------------------------------------------------------------------------
DEMO = PROJECT.parent / "review-demo"   # built by scripts/make_review_demo.py; keep it OUTSIDE the project
CONFIG_PATH: Path | None = None         # None = <repo>/.agentic-review.yml if present, else the defaults
# ---------------------------------------------------------------------------------------------------------------------


def read_env_file(path: Path) -> dict[str, str]:
    values = {}
    if path.is_file():
        for line in path.read_text(encoding="utf-8").splitlines():
            line = line.strip()
            if line and not line.startswith("#") and "=" in line:
                key, _, value = line.partition("=")
                values[key.strip()] = value.strip().strip('"')
    return values


# Preflight only checks that a credential can be found; nothing here sends it anywhere. The value is never printed.
API_KEY = read_env_file(PROJECT / ".env").get("AGENTIC_REVIEW_API_KEY") or "notebook-placeholder"
ENV = {**os.environ, "AGENTIC_REVIEW_API_KEY": API_KEY}

if not DEMO.exists():
    script = PROJECT / "scripts" / "make_review_demo.py"
    print(subprocess.run([sys.executable, str(script), "--dest", str(DEMO)], capture_output=True, text=True).stdout)

print("project:", PROJECT)
print("demo repository:", DEMO)
print("credential in .env:", API_KEY != "notebook-placeholder")

project: d:\Projects\DEPI-industry-project\Pull-request-Reviewer
demo repository: d:\Projects\DEPI-industry-project\review-demo
credential in .env: True


## 1 · Run the graph up to and including `preflight`

`stream_mode="updates"` yields once per node that finished. Leaving the loop after `preflight` stops the graph before
`resolve_change_set` starts: this is the real compiled graph, not a copy of its nodes.

In [3]:
import time

from agentic_review.agents.pipeline_agent import (
    PipelineDeps,
    PipelineRun,
    ReviewRequest,
    build_pipeline_graph,
    utc_now,
)
from agentic_review.agents.review_port import build_review_port
from agentic_review.core.config.credentials import credential_sources


def run_until(node: str, request: ReviewRequest, env: dict[str, str]) -> tuple[PipelineRun, PipelineDeps, list[str]]:
    """Run the pipeline graph and stop once `node` has finished (or the graph ended earlier)."""
    run = PipelineRun(request=request, clock=time.monotonic, now=utc_now)
    deps = PipelineDeps(credentials=credential_sources(env), port_factory=build_review_port)
    finished: list[str] = []
    for update in build_pipeline_graph().stream({"run": run}, context=deps, stream_mode="updates"):
        finished.extend(update)
        if node in update:
            break
    return run, deps, finished


run, deps, finished = run_until("preflight", ReviewRequest(cwd=DEMO, config_path=CONFIG_PATH), ENV)
print("nodes that ran:", finished)
print("stopped before resolve_change_set:", run.change_set is None)

nodes that ran: ['preflight']
stopped before resolve_change_set: True


## 2 · The state after preflight

Preflight either fills in `checked` and `agent` and leaves `result` empty (the run continues), or sets `result` to a
`PreflightFailure` (the run ends; see §5). Everything after `agent` is still empty: later nodes fill it in.

In [4]:
from dataclasses import fields

for f in fields(run):
    value = getattr(run, f.name)
    shown = type(value).__name__ if f.name in ("clock", "now", "timer") else value
    print(f"{f.name:15} {shown!r:.110}")
print("\nstage durations (ms):", run.timer.durations_ms)

request         ReviewRequest(cwd=WindowsPath('d:/Projects/DEPI-industry-project/review-demo'), scope=None, base=None, config_
clock           'builtin_function_or_method'
now             'function'
timer           'StageTimer'
started_at      '2026-09-24T14:11:47Z'
run_started     524160.312
checked         PreflightOk(repo_root=WindowsPath('D:/Projects/DEPI-industry-project/review-demo'), loaded=LoadedSettings(sett
agent           AgentSettings(tool_call_ceiling=25, investigation_tools=False, model='qwen2.5-coder:7b', base_url='http://loca
change_set      None
change_summary  None
line_counts     None
context         None
outcome         None
result          None

stage durations (ms): {'preflight': 31}


In [5]:
checked = run.checked
assert checked is not None, run.result  # preflight failed: see run.result.message

print("repository root:     ", checked.repo_root)
print("scope from the file: ", checked.loaded.scope_from_file)
print("ignored settings:    ", checked.loaded.ignored_settings)
print("\nthe agent settings the review node will use (after any --ceiling override):")
print(json.dumps(run.agent.model_dump(mode="json", by_alias=True), indent=2))

repository root:      D:\Projects\DEPI-industry-project\review-demo
scope from the file:  False
ignored settings:     ()

the agent settings the review node will use (after any --ceiling override):
{
  "toolCallCeiling": 25,
  "investigationTools": false,
  "model": "qwen2.5-coder:7b",
  "baseUrl": "http://localhost:11434/v1",
  "temperature": 0.0,
  "requestTimeoutSeconds": 60,
  "rateLimitRetrySeconds": 60
}


In [6]:
# The whole validated .agentic-review.yml, defaults filled in.
print(json.dumps(checked.loaded.settings.model_dump(mode="json", by_alias=True), indent=2))

{
  "schemaVersion": 1,
  "scope": "working-tree",
  "exclude": [],
  "context": {
    "depth": 1,
    "node": "node"
  },
  "checks": {
    "tests": {
      "command": null,
      "timeoutSeconds": 120
    },
    "typecheck": {
      "command": null,
      "timeoutSeconds": 120
    }
  },
  "agent": {
    "toolCallCeiling": 25,
    "investigationTools": false,
    "model": "qwen2.5-coder:7b",
    "baseUrl": "http://localhost:11434/v1",
    "temperature": 0.0,
    "requestTimeoutSeconds": 60,
    "rateLimitRetrySeconds": 60
  },
  "budget": {
    "maxTokens": 20000
  }
}


## 3 · What a tracer would receive for this state

A tracer gets `{"run": run}` as the node's output and ships it serialized. This uses Langfuse's own serializer, the
one its LangChain handler uses, so what's printed is what would leave the machine. The check below fails loudly if
the credential or any other environment value ever ends up in the state again.

In [7]:
from langfuse._utils.serializer import EventSerializer

shipped = json.dumps({"run": run}, cls=EventSerializer, indent=2)
print(shipped[:4000] + ("\n..." if len(shipped) > 4000 else ""))

{
  "run": {
    "request": {
      "cwd": "d:\\Projects\\DEPI-industry-project\\review-demo",
      "scope": null,
      "base": null,
      "config_path": null,
      "ceiling": null
    },
    "clock": "<builtin_function_or_method>",
    "now": {},
    "timer": {
      "_clock": "<builtin_function_or_method>",
      "durations_ms": {
        "preflight": 31
      }
    },
    "started_at": "2026-09-24T14:11:47Z",
    "run_started": 524160.312,
    "checked": {
      "repo_root": "D:\\Projects\\DEPI-industry-project\\review-demo",
      "loaded": {
        "settings": {
          "schema_version": 1,
          "scope": "working-tree",
          "exclude": [],
          "context": {
            "depth": 1,
            "node": "node"
          },
          "checks": {
            "tests": {
              "command": null,
              "timeout_seconds": 120
            },
            "typecheck": {
              "command": null,
              "timeout_seconds": 120
            }
      

In [9]:
secrets = {name: value for name, value in ENV.items() if any(w in name.upper() for w in ("KEY", "SECRET", "TOKEN"))}
leaked = [name for name, value in secrets.items() if value and len(value) > 3 and value in shipped]
print(f"checked {len(secrets)} secret-looking environment variables:", sorted(secrets))
print("found in the serialized state:", leaked or "none")
assert not leaked

checked 4 secret-looking environment variables: ['AGENTIC_REVIEW_API_KEY', 'GROQ_API_KEY', 'LANGFUSE_PUBLIC_KEY', 'LANGFUSE_SECRET_KEY']
found in the serialized state: none


## 4 · The run context: `PipelineDeps`

Not state. Nodes read it through `runtime.context`, and LangGraph doesn't pass it to callbacks, so no tracer receives
it. It holds only what `credential_sources(env)` kept from the environment, not the environment itself.
`Credential` never shows its value (`repr` and `str` are masked).

In [ ]:
sources = deps.credentials
print("from the environment:", sources.from_env)       # Credential(..., value='***') or None
print("user-level file:     ", sources.user_file, "(exists)" if sources.user_file.is_file() else "(absent)")
print("port factory:        ", deps.port_factory.__name__)

## 5 · When preflight fails

Preflight is the only stage that can stop a run. On failure it sets `result`, and the conditional edge after it routes
to `END`, so no other node runs. Try a directory that isn't a git repository, a missing config file, or an
environment with no credential.

In [10]:
not_a_repo = Path(tempfile.mkdtemp())
cases = {
    "not a git repository": (ReviewRequest(cwd=not_a_repo), ENV),
    "config file missing": (ReviewRequest(cwd=DEMO, config_path=DEMO / "no-such.yml"), ENV),
    "no credential": (ReviewRequest(cwd=DEMO), {k: v for k, v in ENV.items() if k != "AGENTIC_REVIEW_API_KEY"}),
}
for label, (request, env) in cases.items():
    failed, _, ran = run_until("preflight", request, env)
    print(f"--- {label}  (nodes that ran: {ran})")
    print(failed.result.message if failed.result is not None else "preflight passed", end="\n\n")

--- not a git repository  (nodes that ran: ['preflight'])
C:\Users\moham\AppData\Local\Temp\tmp70fl63r_ is not inside a git repository.
Nothing was reviewed. Run the review from inside the repository you want reviewed.

--- config file missing  (nodes that ran: ['preflight'])
Error in d:\Projects\DEPI-industry-project\review-demo\no-such.yml
  the configuration file does not exist
Nothing was reviewed. Fix the field above and re-run.

--- no credential  (nodes that ran: ['preflight'])
No provider credential was found. Looked in:
  the environment variable AGENTIC_REVIEW_API_KEY
  the user-level file C:\Users\moham\AppData\Roaming\agentic-review\credentials.yml (containing 'apiKey: <key>')
Nothing was reviewed. Supply the credential in one of these places and re-run.

